# Qwen3-0.6B: solution vs whole completion (Countdown)

The Qwen counterpart of `Context_Score_Countdown_Llama`'s whole-completion views, for `tab_judges_countdown`. The
completions are the SFT policy's (S) own samples on the 400 grade puzzles (200 four-number + 200 three-number) x 8, T 0.9,
cap 1,024 (`grade_n200x8_t0.9_cap1024_seed0/gen_sft.csv`). Their solution + answer scores under the judge prompt (judge,
S, base model) are already in the GRPO notebook's calibration file (`judge_sum`, `S_sum`, `base_sum`). This notebook adds:
- **gen_full**: S on its own GRPO prompt (puzzle + format block, thinking off), scoring the whole completion + end of turn
- **gen_full_base**: the same sequence under the untrained model (adapter off)

Scored: the 1,766 completions the calibration scored (a solution of at most 128 tokens and an answer; all finished), so
both spans are compared on the same completions. Per view: `mean`, `sum`, `low10`, `min`, token count.
[checks] rescores N_CHECK completions in the judge view and asserts they reproduce the calibration's `S_sum` and
`base_sum`, and that the whole-completion tokens match the generated token counts.
Inference only, no vLLM; a few minutes on any GPU. Cached: a finished OUT_CSV is loaded, not scored again.

Run all: [config] → [drive] → [install] (no restart) → [setup] → [data] → [score] → [checks] → [release]

In [ ]:
# [config] The completions (S's grade file), S's adapter, and the calibration file whose scored rows are rescored.
MODEL, CHAT_KWARGS = "Qwen/Qwen3-0.6B", {"enable_thinking": False}
SFT_RUN = "Qwen3-0.6B-countdown500-ep1-lr0.0001-seed0-tr4-f16-8-w35-30-25-10-imp0x0-mix3x500"
DRIVE = "/content/drive/MyDrive/Consistency"
CODE_DIR = f"{DRIVE}/code/countdown"
S_DIR = f"{DRIVE}/countdown/sft/{SFT_RUN}"
S_ADAPTER = f"{S_DIR}/adapter"
S_GRADE = f"{S_DIR}/grade_n200x8_t0.9_cap1024_seed0/gen_sft.csv"
JUDGE_DIR = f"{S_DIR}/judge_corrupt_empty-cut0.1-cut0.25-cut0.5_unfinished_n4x1000_n3x1000_t0.9_cap2048_seed0/dpo_empty_b0.1_lr2e-05_ep2"
CALIB_SCORES = f"{JUDGE_DIR}/calibration_sum_missmed_S_grade_n200x8_t0.9_cap1024_seed0_scores.csv"   # GRPO notebook [judge]
OUT_DIR = f"{S_DIR}/context_score"
OUT_CSV = f"{OUT_DIR}/context_scores_grade_n200x8_t0.9_cap1024_seed0.csv"
LOW_K = 10
CHUNK = 4                 # sequences per forward pass
N_CHECK = 64              # [checks]: completions rescored in the judge view (S and base) against the calibration
RESCORE = False           # True: score again even if OUT_CSV exists
UNASSIGN_WHEN_DONE = True # [release]

In [ ]:
# [drive] Mount Drive (Qwen is not gated: no HF login) and check the inputs before any GPU work.
import os
from google.colab import drive
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")
for f in (f"{CODE_DIR}/prompts.py", f"{CODE_DIR}/rewards.py", f"{S_ADAPTER}/adapter_model.safetensors", S_GRADE, CALIB_SCORES):
    assert os.path.exists(f), f"missing {f}"
os.makedirs(OUT_DIR, exist_ok=True)
print("inputs found; output:", OUT_CSV, "(exists)" if os.path.exists(OUT_CSV) else "")

In [ ]:
# [install] PEFT for the adapter (no vLLM, no TRL: inference only), on Colab's torch and transformers; no restart.
# Colab's torchao is too old for peft (peft refuses to import): removed.
import importlib.util, subprocess, sys
if importlib.util.find_spec("peft"):
    print("[install] skipped: peft is installed")
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "peft"], check=True)
    print("[install] done")
if importlib.util.find_spec("torchao"):
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=True)
    print("[install] removed torchao")

In [ ]:
# [setup] Tokenizer and chat template (CHAT_KWARGS on every call), the Countdown prompts (Drive), the end-of-turn token,
# and the judge view's text (as GRPO_Countdown_Qwen's [setup] / [rewards]).
import os, sys, re, json, importlib, time
import numpy as np
import pandas as pd
from transformers import AutoTokenizer
sys.path.insert(0, CODE_DIR)
import prompts, rewards as R
for _m in (prompts, R):
    importlib.reload(_m)
from prompts import countdown_prompt, judge_messages

tok = AutoTokenizer.from_pretrained(MODEL)
assert tok.pad_token is not None and tok.pad_token_id != tok.eos_token_id, tok.pad_token

def chat_ids(messages):
    return tok(tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, **CHAT_KWARGS),
               add_special_tokens=False)["input_ids"]

def end_of_turn_id():
    s = tok.apply_chat_template([{"role": "user", "content": "Q"}, {"role": "assistant", "content": "XYZXYZ"}],
                                tokenize=False, **CHAT_KWARGS)
    eot = tok(s.split("XYZXYZ", 1)[1], add_special_tokens=False)["input_ids"][0]
    assert tok.convert_ids_to_tokens(eot) in tok.all_special_tokens, tok.convert_ids_to_tokens(eot)
    return eot

EOT = end_of_turn_id()
judge_text = lambda sol, ans: f"{sol.strip()}\nAnswer: {ans.strip()}"
assert chat_ids([{"role": "user", "content": "x"}]) and tok.decode(chat_ids([{"role": "user", "content": "x"}])).rstrip().endswith("</think>")
print("tokenizer:", MODEL, "| pad", tok.pad_token, "| end of turn", tok.decode([EOT]))

In [ ]:
# [data] S's grade completions, row-aligned with the calibration's scores; keep the rows the calibration scored.
_str = {c: str for c in ("completion", "reasoning", "solution", "answer")}
grade = pd.read_csv(S_GRADE, keep_default_na=False, dtype=_str)
cal = pd.read_csv(CALIB_SCORES)
assert len(grade) == len(cal), (len(grade), len(cal))
for k in ("n_numbers", "puzzle", "numbers", "target", "sample", "n_tokens"):
    assert (grade[k].astype(str).values == cal[k].astype(str).values).all(), f"grade file and calibration differ in {k}"
_t = lambda s: s.astype(str) == "True"
data = grade.assign(scored=_t(cal.scored).values, correct=_t(cal.correct_r).values, finished=_t(grade.finished),
                    S_sum=cal.S_sum.values, base_sum=cal.base_sum.values, judge_sum=cal.judge_sum.values)
data = data[data.scored].reset_index(drop=True)
assert data.finished.all(), "a scored completion did not finish"
data["numbers"] = data.numbers.map(json.loads)
data["group"] = data.n_numbers.astype(str) + "-" + data.puzzle.astype(str)
_g = data.groupby("group").correct.agg(lambda c: c.any() and not c.all())
print(f"{len(data)} scored completions of {len(grade)}; correct {data.correct.mean():.3f}; "
      f"{data.group.nunique()} puzzles, {int(_g.sum())} with both correct and wrong")

In [ ]:
# [score] The base model, S's adapter on (gen_full) or off (gen_full_base): per-token log-probs of the whole completion +
# end of turn after S's GRPO prompt -> mean, sum, low10, min, tokens. Cached in OUT_CSV.
import contextlib
import torch
from transformers import AutoModelForCausalLM
from peft import PeftModel
SCORES = ("mean", "sum", "low10", "min", "tokens")
DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")

def summarize(lp):
    lp = np.asarray(lp, dtype=np.float64)
    return {"mean": lp.mean(), "sum": lp.sum(), "low10": np.sort(lp)[:LOW_K].sum(), "min": lp.min(), "tokens": len(lp)}

def score(model, seqs, adapter_on):
    """[{scores}] for seqs = [(prompt ids, scored ids)], S's adapter on or off."""
    out = [None] * len(seqs)
    order = sorted(range(len(seqs)), key=lambda i: len(seqs[i][0]) + len(seqs[i][1]))
    with torch.no_grad(), (contextlib.nullcontext() if adapter_on else model.disable_adapter()):
        for s in range(0, len(order), CHUNK):
            idx = order[s:s + CHUNK]
            L = max(len(seqs[i][0]) + len(seqs[i][1]) for i in idx)
            ids = torch.full((len(idx), L), tok.pad_token_id, dtype=torch.long)
            attn = torch.zeros_like(ids)
            for j, i in enumerate(idx):
                x = seqs[i][0] + seqs[i][1]
                ids[j, :len(x)], attn[j, :len(x)] = torch.tensor(x), 1
            logits = model(input_ids=ids.to(DEVICE), attention_mask=attn.to(DEVICE)).logits
            for j, i in enumerate(idx):
                p, t = seqs[i]
                lg = logits[j, len(p) - 1:len(p) + len(t) - 1].float()
                lp = torch.log_softmax(lg, -1).gather(-1, torch.tensor(t, device=DEVICE)[:, None])[:, 0]
                out[i] = summarize(lp.cpu().numpy())
            del logits
    return out

def load_model():
    base = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.bfloat16).to(DEVICE).eval()
    return PeftModel.from_pretrained(base, S_ADAPTER, adapter_name="sft").eval()

full_seqs = [(chat_ids(countdown_prompt(n, int(t))), tok(c, add_special_tokens=False)["input_ids"] + [EOT])
             for n, t, c in zip(data.numbers, data.target, data.completion)]
if os.path.exists(OUT_CSV) and not RESCORE:
    sc = pd.read_csv(OUT_CSV)
    assert len(sc) == len(data) and (sc.puzzle.values == data.puzzle.values).all() and (sc["sample"].values == data["sample"].values).all()
    for c in sc.columns:
        if c.startswith("gen_full"):
            data[c] = sc[c].values
    print(f"[score] loaded {OUT_CSV}")
else:
    model = load_model()
    t0 = time.time()
    for view, on in (("gen_full", True), ("gen_full_base", False)):
        res = pd.DataFrame(score(model, full_seqs, on))
        for m in SCORES:
            data[f"{view}_{m}"] = res[m].values
        print(f"[score] {view}: {(time.time() - t0) / 60:.1f} min")
    cols = ["n_numbers", "puzzle", "sample", "correct", "n_tokens"] + [c for c in data.columns if c.startswith("gen_full")]
    data[cols].to_csv(OUT_CSV, index=False)
    print(f"[score] saved {OUT_CSV}")

In [ ]:
# [checks] (1) The whole-completion token count (retokenized text + end of turn) vs vLLM's generated count (which
# includes the stop token): equal for (almost) every row. (2) N_CHECK completions rescored in the judge view (judge prompt + solution + "Answer: ..." + end of turn,
# S and the untrained model) reproduce the calibration's S_sum and base_sum (bf16 sums over ~70 tokens, other batching and
# GPU: median |diff| < 3% of the value and correlation > 0.995; a wrong prompt or text is far off). (3) Within-puzzle win rates of correct over wrong (ties 1/2), sum score, both spans.
_eq = (data.gen_full_tokens == data.n_tokens.astype(int)).mean()
print(f"[checks] whole-completion tokens (text + end of turn) = generated tokens: {_eq:.3f} of rows "
      f"(max |diff| {(data.gen_full_tokens - data.n_tokens.astype(int)).abs().max():.0f})")
assert _eq > 0.95, "the retokenized completions do not match the generated token counts"
_chk = data.sample(min(N_CHECK, len(data)), random_state=0)
_jseqs = [(chat_ids(judge_messages(list(n), int(t))), tok(judge_text(s, a), add_special_tokens=False)["input_ids"] + [EOT])
          for n, t, s, a in zip(_chk.numbers, _chk.target, _chk.solution, _chk.answer)]
if "model" not in globals():
    model = load_model()
for name, on, ref in (("S", True, "S_sum"), ("base", False, "base_sum")):
    new = np.array([r["sum"] for r in score(model, _jseqs, on)])
    d = np.abs(new - _chk[ref].values)
    rel = np.median(d) / np.median(np.abs(_chk[ref].values))
    r = np.corrcoef(new, _chk[ref].values)[0, 1]
    print(f"[checks] judge view, {name}: median |sum - calibration {ref}| {np.median(d):.4f} ({rel:.2%}), max {d.max():.4f}, "
          f"correlation {r:.4f}")
    assert rel < 0.03 and r > 0.995, f"the judge view does not reproduce the calibration's {ref}"

def win(df, col):
    w = t = sep = mixed = 0
    for _, g in df.groupby("group"):
        c, x = g.loc[g.correct, col].values, g.loc[~g.correct, col].values
        if len(c) and len(x):
            dd = c[:, None] - x[None, :]
            w, t, mixed, sep = w + (dd > 0).sum() + 0.5 * (dd == 0).sum(), t + dd.size, mixed + 1, sep + (c.min() > x.max())
    return (w / t, sep / mixed, t) if t else (float("nan"), float("nan"), 0)

rows = [(lab, col, *win(data, col)) for lab, col in
        (("base, solution + answer", "base_sum"), ("base, whole completion", "gen_full_base_sum"),
         ("S, solution + answer", "S_sum"), ("S, whole completion", "gen_full_sum"), ("judge, solution + answer", "judge_sum"))]
print(pd.DataFrame(rows, columns=["scorer, span", "column", "pairs", "groups separated", "n pairs"]).to_string(index=False))

In [ ]:
# [release] All results are on Drive: release the runtime if UNASSIGN_WHEN_DONE (a failed cell stops Run all before this).
if UNASSIGN_WHEN_DONE:
    from google.colab import runtime
    runtime.unassign()